# 01 · Coleta de Dados (DataSUS)

**Objetivo:** Baixar arquivos `.dbc` diretamente do servidor FTP público do DataSUS e convertê-los para `.csv`, alimentando as etapas seguintes do projeto.

**Inputs:** Nenhum (acesso direto ao FTP do DataSUS via internet).

**Outputs gerados:**
- `data/raw/SIH/` — arquivos `.dbc` brutos do Sistema de Informações Hospitalares
- `data/raw/CNES/` — arquivos `.dbc` brutos do Cadastro Nacional de Estabelecimentos
- `data/input/SIH/` — arquivos `.csv` convertidos do SIH
- `data/input/CNES/` — arquivos `.csv` convertidos do CNES

**Sistemas do DATASUS utilizados:**

| Sigla | Nome completo |
|-------|---------------|
| `SIH`  | Sistema de Informações Hospitalares |
| `CNES` | Cadastro Nacional de Estabelecimentos de Saúde |

<br/>

> **Nota:** O `SIM` (Sistema de Informação sobre Mortalidade) está disponível nos utilitários mas não é utilizado neste projeto.

## 0. Configuração do Ambiente

In [11]:
import sys
import os
import pandas as pd
from pathlib import Path

In [12]:
# Garante que a raiz do projeto está no sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [13]:
# Importa utilitários do projeto
from src.utils.download_data_from_datasus import download_data, download_dicionarios
from src.utils.converter_dbc_para_csv import converter_dbc_para_csv_lote

print("Utilitários importados")

Utilitários importados


## 1. Coleta do SIH (Sistema de Informações Hospitalares)

Baixamos os registros de AIH (Autorização de Internação Hospitalar) para o estado e ano definidos abaixo.
O fluxo é:
1. Download do `.dbc` do FTP do DataSUS → `data/raw/SIH/`
2. Conversão `.dbc → .csv` → `data/input/SIH/`

In [14]:
# ── Parâmetros de coleta ─────────────────────────────────────────────
ESTADOS_SIH = ["SP"]    # lista de UFs a baixar
ANOS_SIH    = [2025]    # anos de referência
SISTEMA_SIH = "SIH"     # identificador do sistema DataSUS

# ── Diretórios ───────────────────────────────────────────────────────
PASTA_RAW_SIH   = Path(ROOT, "data", "raw",   SISTEMA_SIH)
PASTA_INPUT_SIH = Path(ROOT, "data", "input", SISTEMA_SIH)

PASTA_RAW_SIH.mkdir(parents=True, exist_ok=True)
PASTA_INPUT_SIH.mkdir(parents=True, exist_ok=True)

print(f"RAW   : {PASTA_RAW_SIH}")
print(f"INPUT : {PASTA_INPUT_SIH}")

RAW   : /home/carolina/Documents/TCC Documentos/TCC/data/raw/SIH
INPUT : /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH


In [ ]:
# Download dos arquivos .dbc do SIH
print("Iniciando download do SIH...")
download_data(
    estados=ESTADOS_SIH,
    anos=ANOS_SIH,
    sistema=SISTEMA_SIH,
    download_path=str(PASTA_RAW_SIH),
)
print("Download concluído.")

In [ ]:
# Conversão .dbc → .csv
print("Convertendo arquivos SIH de .dbc para .csv...")
converter_dbc_para_csv_lote(str(PASTA_RAW_SIH), str(PASTA_INPUT_SIH))



In [23]:
# Validação: lista arquivos gerados
csvs_sih = sorted(PASTA_INPUT_SIH.glob("*.csv"))
print(f"\nArquivos CSV gerados ({len(csvs_sih)}):")
for f in csvs_sih:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")


Arquivos CSV gerados (12):
  rdsp2501.csv  (106533 KB)
  rdsp2502.csv  (103466 KB)
  rdsp2503.csv  (110689 KB)
  rdsp2504.csv  (110343 KB)
  rdsp2505.csv  (116755 KB)
  rdsp2506.csv  (110543 KB)
  rdsp2507.csv  (114801 KB)
  rdsp2508.csv  (114002 KB)
  rdsp2509.csv  (113085 KB)
  rdsp2510.csv  (115507 KB)
  rdsp2511.csv  (107749 KB)
  rdsp2512.csv  (103407 KB)


## 2. Coleta do CNES (Cadastro Nacional de Estabelecimentos de Saúde)

Baixamos os arquivos das tabelas auxiliares do CNES:
- **ST** — Estabelecimentos (tabela principal)
- **LT** — Leitos
- **EQ** — Equipamentos
- **SR** — Serviços Especializados
- **HB** — Habilitações

Esses dados serão integrados com o SIH no notebook `02_eda.ipynb`.

In [15]:
# ── Parâmetros de coleta ─────────────────────────────────────────────
ESTADOS_CNES = ["SP"]   # lista de UFs a baixar
ANOS_CNES    = [2025]   # anos de referência
MESES_CNES = [12]
BASES_CNES = ["ST","LT", "EQ", "HB", "SR"]
SISTEMA_CNES = "CNES"   # identificador do sistema DataSUS

# ── Diretórios ───────────────────────────────────────────────────────
PASTA_RAW_CNES   = Path(ROOT, "data", "raw",   SISTEMA_CNES)
PASTA_INPUT_CNES = Path(ROOT, "data", "input", SISTEMA_CNES)

PASTA_RAW_CNES.mkdir(parents=True, exist_ok=True)
PASTA_INPUT_CNES.mkdir(parents=True, exist_ok=True)

print(f"RAW   : {PASTA_RAW_CNES}")
print(f"INPUT : {PASTA_INPUT_CNES}")

RAW   : /home/carolina/Documents/TCC Documentos/TCC/data/raw/CNES
INPUT : /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


In [ ]:
# Download dos arquivos .dbc do CNES
print("Iniciando download do CNES...")
download_data(
    estados=ESTADOS_CNES,
    anos=ANOS_CNES,
    meses=MESES_CNES,
    sistema=SISTEMA_CNES,
    bases_cnes=BASES_CNES,
    download_path=str(PASTA_RAW_CNES),
)
print("Download concluído.")

In [ ]:
# Conversão .dbc → .csv
print("Convertendo arquivos CNES de .dbc para .csv...")
converter_dbc_para_csv_lote(str(PASTA_RAW_CNES), str(PASTA_INPUT_CNES))


In [ ]:
# Validação: lista arquivos gerados
csvs_cnes = sorted(PASTA_INPUT_CNES.glob("*.csv"))
print(f"\nArquivos CSV gerados ({len(csvs_cnes)}):")
for f in csvs_cnes:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

## 4. Tradução dos Códigos das Tabelas

O DataSUS armazena os dados como **códigos numéricos** (ex: `1`, `3`, `M`). Esta seção
traduz esses códigos para **descrições legíveis** usando os dicionários `.cnv` e as
instruções `.def` baixados do FTP do DataSUS.

**Como funciona:**
1. O arquivo `.def` indica quais colunas têm código e qual `.cnv` usar para cada uma
2. O arquivo `.cnv` contém o mapeamento `CÓDIGO → DESCRIÇÃO`
3. Para cada coluna com dicionário é criada uma coluna `COLUNA_DESC` com o texto traduzido

**Resultado:** arquivos salvos em `data/interim/` com colunas `_DESC` adicionadas.

### 4.0 Download dos dicionários (executar apenas uma vez)

Os dicionários `.def` e `.cnv` são baixados do FTP do DataSUS e salvos em `data/external/`.
Se os arquivos já existirem localmente este passo pode ser pulado.

In [16]:
# Pasta destino ancorada em ROOT para não depender do diretório de trabalho do notebook
PASTA_EXTERNAL_SIH = Path(ROOT, "data", "external", "SIH")
PASTA_EXTERNAL_CNES = Path(ROOT, "data", "external", "CNES")
PASTA_EXTERNAL_SIH.mkdir(parents=True, exist_ok=True)
PASTA_EXTERNAL_CNES.mkdir(parents=True, exist_ok=True)

In [ ]:


# Dicionários do CNES
download_dicionarios("CNES", str(PASTA_EXTERNAL_CNES))

# Dicionários do SIH
download_dicionarios("SIH", str(PASTA_EXTERNAL_SIH))

print(f"\nDicionários salvos")

Conectando ao FTP para buscar tabelas de CNES: ftp.datasus.gov.br
Arquivo encontrado: TAB_CNES.zip. Baixando para a memória...
Extraindo dicionários (.cnv) e configurações (.def)...

Processo concluído! Arquivos salvos em: /home/carolina/Documents/TCC Documentos/TCC/data/external/CNES
Conectando ao FTP para buscar tabelas de SIH: ftp.datasus.gov.br
Arquivo encontrado: TAB_SIH.zip. Baixando para a memória...
Extraindo dicionários (.cnv) e configurações (.def)...

Processo concluído! Arquivos salvos em: /home/carolina/Documents/TCC Documentos/TCC/data/external/SIH

Dicionários salvos


### 4.1 Configuração dos caminhos e importação dos utilitários

In [ ]:
from src.utils.information_translation import mapear_colunas_def, traduzir_csv_datasus

# Caminhos dos dicionários — separados por sistema (CNES e SIH)
PASTA_DEF_CNES  = PASTA_EXTERNAL_CNES                    # arquivos .def do CNES
PASTA_CNV_CNES  = PASTA_EXTERNAL_CNES / "CNV"            # arquivos .cnv do CNES
PASTA_DEF_SIH   = PASTA_EXTERNAL_SIH                     # arquivos .def do SIH
PASTA_CNV_SIH   = PASTA_EXTERNAL_SIH / "CNV"             # arquivos .cnv do SIH

# Pasta de saída dos CSVs traduzidos
PASTA_INTERIM = Path(ROOT, "data", "interim")
PASTA_INTERIM.mkdir(parents=True, exist_ok=True)

print(f"DEF CNES : {PASTA_DEF_CNES}")
print(f"CNV CNES : {PASTA_CNV_CNES}")
print(f"DEF SIH  : {PASTA_DEF_SIH}")
print(f"CNV SIH  : {PASTA_CNV_SIH}")
print(f"Saída    : {PASTA_INTERIM}")

DEF CNES : /home/carolina/Documents/TCC Documentos/TCC/data/external/CNES
CNV CNES : /home/carolina/Documents/TCC Documentos/TCC/data/external/CNES/CNV
DEF SIH  : /home/carolina/Documents/TCC Documentos/TCC/data/external/SIH
CNV SIH  : /home/carolina/Documents/TCC Documentos/TCC/data/external/SIH/CNV
Saída    : /home/carolina/Documents/TCC Documentos/TCC/data/interim


### 4.2 Tradução das tabelas CNES

Cada base do CNES tem seu próprio `.def`:

| Sigla | Tabela                   | Arquivo .def                         |
|-------|--------------------------|--------------------------------------|
| `HB`  | Habilitações             | `Habilitacao.def`                    |
| `LT`  | Leitos por Especialidade | `Leitos_Especialidade.def`           |
| `EQ`  | Equipamentos             | `Equipamento.def`                    |
| `SR`  | Serviços Especializados  | `Servico_Especializado_200803_.def`  |
| `ST`  | Estabelecimentos         | `Estabelecimento.def`                |

In [18]:
# Mapeamento: prefixo do CSV → arquivo .def correspondente
MAPA_DEF_CNES = {
    "hb": "Habilitacao.def",
    "lt": "Leitos_Especialidade.def",
    "eq": "Equipamento.def",
    "sr": "Servico_Especializado_200803_.def",
    "st": "Estabelecimento.def",
}

for prefixo, nome_def in MAPA_DEF_CNES.items():
    csvs = sorted(PASTA_INPUT_CNES.glob(f"{prefixo}*.csv"))
    if not csvs:
        print(f"[AVISO] Nenhum CSV encontrado para '{prefixo}' em {PASTA_INPUT_CNES}")
        continue

    arquivo_def = PASTA_DEF_CNES / nome_def
    mapa = mapear_colunas_def(str(arquivo_def))

    if not mapa:
        print(f"[AVISO] Nenhuma coluna mapeável encontrada em {nome_def}")
        continue

    print(f"\n── {prefixo.upper()} ({nome_def}) ──")
    print(f"   Colunas com dicionário: {list(mapa.keys())}")

    for csv_path in csvs:
        nome_saida = f"cnes_{prefixo}_{csv_path.stem}_traduzido.csv"
        caminho_saida = PASTA_INTERIM / nome_saida
        traduzir_csv_datasus(
            caminho_csv=str(csv_path),
            mapa_diretrizes=mapa,
            pasta_cnv=str(PASTA_CNV_CNES),
            caminho_salvar=str(caminho_saida),
        )

print("\n✓ Tradução CNES concluída.")


── HB (Habilitacao.def) ──
   Colunas com dicionário: ['MAPORTAR', 'CMPT_INI', 'CMPT_FIM', 'COMPETEN', 'TPGESTAO', 'NIV_DEP', 'PF_PJ', 'NATUREZA', 'NAT_JUR', 'RETENCAO', 'NIV_HIER', 'ESFERA_A', 'ATIVIDAD', 'TP_UNID', 'TURNO_AT', 'TP_PREST', 'VINC_SUS', 'CODUFMUN']
Traduzindo a coluna 'CODUFMUN' com o dicionário 'br_extrpobrez.cnv'...
Traduzindo a coluna 'TPGESTAO' com o dicionário 'TPGESTAO.CNV'...
Traduzindo a coluna 'PF_PJ' com o dicionário 'TP_PFPJ.CNV'...
Traduzindo a coluna 'NIV_DEP' com o dicionário 'NIVELDEP.CNV'...
Traduzindo a coluna 'RETENCAO' com o dicionário 'RETENCAO.CNV'...
Traduzindo a coluna 'ATIVIDAD' com o dicionário 'Ativ_Ens.CNV'...
Traduzindo a coluna 'NATUREZA' com o dicionário 'NATUREZA.CNV'...
Traduzindo a coluna 'TP_UNID' com o dicionário 'TP_ESTAB.CNV'...
Traduzindo a coluna 'NIV_HIER' com o dicionário 'NIV_HIER.CNV'...
Traduzindo a coluna 'TP_PREST' com o dicionário 'TIPOPRES.CNV'...
Traduzindo a coluna 'CMPT_INI' com o dicionário 'COMPT.CNV'...
Traduzindo a

/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:84: DtypeWarning: Columns (0: REGSAUDE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_csv)


Traduzindo a coluna 'NATUREZA' com o dicionário 'NATUREZA.CNV'...
Traduzindo a coluna 'TP_UNID' com o dicionário 'TP_ESTAB.CNV'...
Traduzindo a coluna 'NIV_HIER' com o dicionário 'NIV_HIER.CNV'...
Traduzindo a coluna 'CARACTER' com o dicionário 'Srv_Caract.CNV'...
Traduzindo a coluna 'AMB_NSUS' com o dicionário 'AMB_HOSP.CNV'...
Traduzindo a coluna 'COMPETEN' com o dicionário 'COMPETEN.CNV'...
Traduzindo a coluna 'NAT_JUR' com o dicionário 'RETENCAO.CNV'...

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_sr_srsp2512_traduzido.csv

── ST (Estabelecimento.def) ──
   Colunas com dicionário: ['COMPETEN', 'TPGESTAO', 'NIV_DEP', 'PF_PJ', 'CLIENTEL', 'NATUREZA', 'NAT_JUR', 'RETENCAO', 'COD_IR', 'NIV_HIER', 'ESFERA_A', 'ATIVIDAD', 'TP_UNID', 'TURNO_AT', 'TP_PREST', 'DT_EXPED', 'ORGEXPED', 'AV_ACRED', 'CLASAVAL', 'VINC_SUS', 'DT_PUBLM', 'DT_PUBLE', 'GESPRG1E', 'GESPRG2E', 'GESPRG4E', 'GESPRG5E', 'GESPRG6E', 'GESPRG3E', 'SERAP01P', 'SERAP

/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:84: DtypeWarning: Columns (0: CO_BANCO, 1: CO_AGENC, 2: C_CORREN) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_csv)
/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[nova_coluna_desc] = df[coluna_csv].map(dicionario_traducao)
/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = 

Traduzindo a coluna 'CODUFMUN' com o dicionário 'br_extrpobrez.cnv'...
Traduzindo a coluna 'PF_PJ' com o dicionário 'TP_PFPJ.CNV'...
Traduzindo a coluna 'NIV_DEP' com o dicionário 'NIVELDEP.CNV'...
Traduzindo a coluna 'COD_IR' com o dicionário 'RETENMAN.CNV'...
Traduzindo a coluna 'TPGESTAO' com o dicionário 'TPGESTAO.CNV'...
Traduzindo a coluna 'RETENCAO' com o dicionário 'RETENCAO.CNV'...
Traduzindo a coluna 'ATIVIDAD' com o dicionário 'Ativ_Ens.CNV'...
Traduzindo a coluna 'NATUREZA' com o dicionário 'NATUREZA.CNV'...
Traduzindo a coluna 'TP_UNID' com o dicionário 'TP_ESTAB.CNV'...
Traduzindo a coluna 'NIV_HIER' com o dicionário 'NIV_HIER.CNV'...
Traduzindo a coluna 'TP_PREST' com o dicionário 'TIPOPRES.CNV'...
Traduzindo a coluna 'DT_PUBLM' com o dicionário 'ANO_MES.CNV'...
Traduzindo a coluna 'DT_PUBLE' com o dicionário 'ANO_MES.CNV'...
Traduzindo a coluna 'DT_EXPED' com o dicionário 'ANO_MES.CNV'...


...

Traduzindo a coluna 'SERAP04P' com o dicionário 'Srv_AP04.CNV'...
Traduzindo a coluna 'SERAP05P' com o dicionário 'Srv_AP05.CNV'...
Traduzindo a coluna 'SERAP06P' com o dicionário 'Srv_AP06.CNV'...
Traduzindo a coluna 'SERAP07P' com o dicionário 'Srv_AP07.CNV'...
Traduzindo a coluna 'SERAP08P' com o dicionário 'Srv_AP08.CNV'...
Traduzindo a coluna 'SERAP09P' com o dicionário 'Srv_AP09.CNV'...
Traduzindo a coluna 'SERAP10P' com o dicionário 'Srv_AP10.CNV'...
Traduzindo a coluna 'SERAP11P' com o dicionário 'Srv_AP11.CNV'...
Traduzindo a coluna 'RES_BIOL' com o dicionário 'RES_BIOL.CNV'...
Traduzindo a coluna 'RES_QUIM' com o dicionário 'RES_QUIM.CNV'...
Traduzindo a coluna 'RES_RADI' com o dicionário 'RES_RADI.CNV'...
Traduzindo a coluna 'RES_COMU' com o dicionário 'RES_COMU.CNV'...


/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[nova_coluna_desc] = df[coluna_csv].map(dicionario_traducao)
/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[nova_coluna_desc] = df[coluna_csv].map(dicionario_traducao)
/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calli

Traduzindo a coluna 'COMISS01' com o dicionário 'COM_ETMD.CNV'...
Traduzindo a coluna 'COMISS02' com o dicionário 'COM_ETEF.CNV'...
Traduzindo a coluna 'COMISS03' com o dicionário 'COM_FMTR.CNV'...
Traduzindo a coluna 'COMISS04' com o dicionário 'COM_CTIH.CNV'...
Traduzindo a coluna 'COMISS05' com o dicionário 'COM_APCT.CNV'...
Traduzindo a coluna 'COMISS06' com o dicionário 'COM_CIPA.CNV'...
Traduzindo a coluna 'COMISS07' com o dicionário 'COM_RVPR.CNV'...
Traduzindo a coluna 'COMISS08' com o dicionário 'COM_RDME.CNV'...
Traduzindo a coluna 'COMISS09' com o dicionário 'COM_ANOB.CNV'...
Traduzindo a coluna 'COMISS10' com o dicionário 'COM_IVEP.CNV'...
Traduzindo a coluna 'COMISS11' com o dicionário 'COM_NTFD.CNV'...
Traduzindo a coluna 'COMISS12' com o dicionário 'COM_CTZV.CNV'...


/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[nova_coluna_desc] = df[coluna_csv].map(dicionario_traducao)
/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[nova_coluna_desc] = df[coluna_csv].map(dicionario_traducao)
/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calli

Traduzindo a coluna 'COMPETEN' com o dicionário 'COMPETEN.CNV'...
Traduzindo a coluna 'NAT_JUR' com o dicionário 'ESFERAJURC.CNV'...

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_st_stsp2512_traduzido.csv

✓ Tradução CNES concluída.


### 4.3 Tradução do SIH

In [19]:
# Localiza o .def do SIH — tenta extensão em maiúsculo e minúsculo (Linux é case-sensitive)
defs_sih = (
    list(PASTA_DEF_SIH.glob("RD*.def")) +
    list(PASTA_DEF_SIH.glob("RD*.DEF")) +
    list(PASTA_DEF_SIH.glob("AIH*.def")) +
    list(PASTA_DEF_SIH.glob("AIH*.DEF"))
)

if not defs_sih:
    print("[AVISO] Arquivo .def do SIH não encontrado em", PASTA_DEF_SIH)
    print("Arquivos .def disponíveis:", [f.name for f in PASTA_DEF_SIH.iterdir() if f.suffix.upper() == ".DEF"])
else:
    arquivo_def_sih = defs_sih[0]
    mapa_sih = mapear_colunas_def(str(arquivo_def_sih))
    print(f"Usando .def: {arquivo_def_sih.name}")
    print(f"Colunas com dicionário ({len(mapa_sih)}): {list(mapa_sih.keys())}")

    for csv_path in sorted(PASTA_INPUT_SIH.glob("*.csv")):
        nome_saida = f"sih_{csv_path.stem}_traduzido.csv"
        caminho_saida = PASTA_INTERIM / nome_saida
        traduzir_csv_datasus(
            caminho_csv=str(csv_path),
            mapa_diretrizes=mapa_sih,
            pasta_cnv=str(PASTA_CNV_SIH),
            caminho_salvar=str(caminho_saida),
        )

    print("\n✓ Tradução SIH concluída.")

Usando .def: RD2008.DEF
Colunas com dicionário (56): ['MUNIC_MOV', 'CNES', 'NATUREZA', 'NAT_JUR', 'GESTAO', 'UF_ZI', 'ANO_CMPT', 'MES_CMPT', 'SEXO', 'COD_IDADE', 'RACA_COR', 'ETNIA', 'MUNIC_RES', 'NACIONAL', 'CNAER', 'VINCPREV', 'ESPEC', 'N_AIH', 'IDENT', 'SEQ_AIH5', 'CAR_INT', 'UTI_MES_TO', 'MARCA_UTI', 'MARCA_UCI', 'DT_INTER', 'DT_SAIDA', 'DIAS_PERM', 'COBRANCA', 'MORTE', 'FINANC', 'FAEC_TP', 'REGCT', 'COMPLEX', 'PROC_REA', 'IND_VDRL', 'INFEHOSP', 'DIAG_PRINC', 'TPDISEC1', 'TPDISEC2', 'TPDISEC3', 'TPDISEC4', 'TPDISEC5', 'TPDISEC6', 'TPDISEC7', 'TPDISEC8', 'TPDISEC9', 'DIAG_SECUN', 'CID_ASSO', 'CID_MORTE', 'NUM_FILHOS', 'INSTRU', 'CID_NOTIF', 'CONTRACEP1', 'CONTRACEP2', 'GESTRISCO', 'INSC_PN']


/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:84: DtypeWarning: Columns (0: ETNIA, 1: AUD_JUST, 2: SIS_JUST, 3: DIAGSEC2, 4: DIAGSEC3, 5: DIAGSEC4, 6: DIAGSEC5, 7: DIAGSEC6, 8: DIAGSEC7, 9: DIAGSEC8, 10: DIAGSEC9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_csv)


Traduzindo a coluna 'UF_ZI' com o dicionário 'br_municgestor.cnv'...
Traduzindo a coluna 'ANO_CMPT' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'MES_CMPT' com o dicionário 'MESES.CNV'...
Traduzindo a coluna 'ESPEC' com o dicionário 'LEITOS.CNV'...
Traduzindo a coluna 'N_AIH' com o dicionário 'IDENTIFIC.CNV'...
Traduzindo a coluna 'IDENT' com o dicionário 'IDENT.CNV'...
Traduzindo a coluna 'MUNIC_RES' com o dicionário 'br_pndr.cnv'...
Traduzindo a coluna 'SEXO' com o dicionário 'SEXO.CNV'...
Traduzindo a coluna 'UTI_MES_TO' com o dicionário 'DIARIASUTI.CNV'...
Traduzindo a coluna 'MARCA_UTI' com o dicionário 'MARCAUTI.CNV'...
Traduzindo a coluna 'PROC_REA' com o dicionário 'PROCOBS2b.CNV'...
Traduzindo a coluna 'DT_INTER' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DT_SAIDA' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DIAG_PRINC' com o dicionário 'DIAGPACTO.CNV'...
Traduzindo a coluna 'DIAG_SECUN' com o dicionário 'CID10_22.CNV'...
Traduzindo a coluna 'CO

/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:84: DtypeWarning: Columns (0: ETNIA, 1: AUD_JUST, 2: SIS_JUST, 3: DIAGSEC2, 4: DIAGSEC3, 5: DIAGSEC4, 6: DIAGSEC5, 7: DIAGSEC6, 8: DIAGSEC7, 9: DIAGSEC8, 10: DIAGSEC9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_csv)


Traduzindo a coluna 'UF_ZI' com o dicionário 'br_municgestor.cnv'...
Traduzindo a coluna 'ANO_CMPT' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'MES_CMPT' com o dicionário 'MESES.CNV'...
Traduzindo a coluna 'ESPEC' com o dicionário 'LEITOS.CNV'...
Traduzindo a coluna 'N_AIH' com o dicionário 'IDENTIFIC.CNV'...
Traduzindo a coluna 'IDENT' com o dicionário 'IDENT.CNV'...
Traduzindo a coluna 'MUNIC_RES' com o dicionário 'br_pndr.cnv'...
Traduzindo a coluna 'SEXO' com o dicionário 'SEXO.CNV'...
Traduzindo a coluna 'UTI_MES_TO' com o dicionário 'DIARIASUTI.CNV'...
Traduzindo a coluna 'MARCA_UTI' com o dicionário 'MARCAUTI.CNV'...
Traduzindo a coluna 'PROC_REA' com o dicionário 'PROCOBS2b.CNV'...
Traduzindo a coluna 'DT_INTER' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DT_SAIDA' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DIAG_PRINC' com o dicionário 'DIAGPACTO.CNV'...
Traduzindo a coluna 'DIAG_SECUN' com o dicionário 'CID10_22.CNV'...
Traduzindo a coluna 'CO

...

Traduzindo a coluna 'UF_ZI' com o dicionário 'br_municgestor.cnv'...
Traduzindo a coluna 'ANO_CMPT' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'MES_CMPT' com o dicionário 'MESES.CNV'...
Traduzindo a coluna 'ESPEC' com o dicionário 'LEITOS.CNV'...
Traduzindo a coluna 'N_AIH' com o dicionário 'IDENTIFIC.CNV'...
Traduzindo a coluna 'IDENT' com o dicionário 'IDENT.CNV'...
Traduzindo a coluna 'MUNIC_RES' com o dicionário 'br_pndr.cnv'...
Traduzindo a coluna 'SEXO' com o dicionário 'SEXO.CNV'...
Traduzindo a coluna 'UTI_MES_TO' com o dicionário 'DIARIASUTI.CNV'...
Traduzindo a coluna 'MARCA_UTI' com o dicionário 'MARCAUTI.CNV'...
Traduzindo a coluna 'PROC_REA' com o dicionário 'PROCOBS2b.CNV'...
Traduzindo a coluna 'DT_INTER' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DT_SAIDA' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DIAG_PRINC' com o dicionário 'DIAGPACTO.CNV'...
Traduzindo a coluna 'DIAG_SECUN' com o dicionário 'CID10_22.CNV'...
Traduzindo a coluna 'CO

/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:84: DtypeWarning: Columns (0: ETNIA, 1: SIS_JUST, 2: DIAGSEC2, 3: DIAGSEC3, 4: DIAGSEC4, 5: DIAGSEC5, 6: DIAGSEC6, 7: DIAGSEC7, 8: DIAGSEC8, 9: DIAGSEC9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_csv)


Traduzindo a coluna 'UF_ZI' com o dicionário 'br_municgestor.cnv'...
Traduzindo a coluna 'ANO_CMPT' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'MES_CMPT' com o dicionário 'MESES.CNV'...
Traduzindo a coluna 'ESPEC' com o dicionário 'LEITOS.CNV'...
Traduzindo a coluna 'N_AIH' com o dicionário 'IDENTIFIC.CNV'...
Traduzindo a coluna 'IDENT' com o dicionário 'IDENT.CNV'...
Traduzindo a coluna 'MUNIC_RES' com o dicionário 'br_pndr.cnv'...
Traduzindo a coluna 'SEXO' com o dicionário 'SEXO.CNV'...
Traduzindo a coluna 'UTI_MES_TO' com o dicionário 'DIARIASUTI.CNV'...
Traduzindo a coluna 'MARCA_UTI' com o dicionário 'MARCAUTI.CNV'...
Traduzindo a coluna 'PROC_REA' com o dicionário 'PROCOBS2b.CNV'...
Traduzindo a coluna 'DT_INTER' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DT_SAIDA' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DIAG_PRINC' com o dicionário 'DIAGPACTO.CNV'...
Traduzindo a coluna 'DIAG_SECUN' com o dicionário 'CID10_22.CNV'...
Traduzindo a coluna 'CO

/home/carolina/Documents/TCC Documentos/TCC/src/utils/traducao_csvs.py:84: DtypeWarning: Columns (0: ETNIA, 1: AUD_JUST, 2: SIS_JUST, 3: DIAGSEC2, 4: DIAGSEC3, 5: DIAGSEC4, 6: DIAGSEC5, 7: DIAGSEC6, 8: DIAGSEC7, 9: DIAGSEC8, 10: DIAGSEC9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_csv)


Traduzindo a coluna 'UF_ZI' com o dicionário 'br_municgestor.cnv'...
Traduzindo a coluna 'ANO_CMPT' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'MES_CMPT' com o dicionário 'MESES.CNV'...
Traduzindo a coluna 'ESPEC' com o dicionário 'LEITOS.CNV'...
Traduzindo a coluna 'N_AIH' com o dicionário 'IDENTIFIC.CNV'...
Traduzindo a coluna 'IDENT' com o dicionário 'IDENT.CNV'...
Traduzindo a coluna 'MUNIC_RES' com o dicionário 'br_pndr.cnv'...
Traduzindo a coluna 'SEXO' com o dicionário 'SEXO.CNV'...
Traduzindo a coluna 'UTI_MES_TO' com o dicionário 'DIARIASUTI.CNV'...
Traduzindo a coluna 'MARCA_UTI' com o dicionário 'MARCAUTI.CNV'...
Traduzindo a coluna 'PROC_REA' com o dicionário 'PROCOBS2b.CNV'...
Traduzindo a coluna 'DT_INTER' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DT_SAIDA' com o dicionário 'ANOMESC.CNV'...
Traduzindo a coluna 'DIAG_PRINC' com o dicionário 'DIAGPACTO.CNV'...
Traduzindo a coluna 'DIAG_SECUN' com o dicionário 'CID10_22.CNV'...
Traduzindo a coluna 'CO

### 4.4 Verificação do resultado

Mostra as colunas `_DESC` geradas em um arquivo de exemplo para confirmar que a tradução funcionou.

In [20]:
arquivos_traduzidos = sorted(PASTA_INTERIM.glob("*.csv"))
print(f"{len(arquivos_traduzidos)} arquivo(s) traduzido(s) em {PASTA_INTERIM}\n")

if arquivos_traduzidos:
    exemplo = arquivos_traduzidos[0]
    df_ex = pd.read_csv(exemplo, nrows=3)
    colunas_desc = [c for c in df_ex.columns if c.endswith("_DESC")]
    print(f"Exemplo: {exemplo.name}")
    print(f"Colunas _DESC geradas ({len(colunas_desc)}): {colunas_desc}\n")
    display(df_ex[colunas_desc].head(3))

17 arquivo(s) traduzido(s) em /home/carolina/Documents/TCC Documentos/TCC/data/interim

Exemplo: cnes_eq_eqsp2512_traduzido.csv
Colunas _DESC geradas (13): ['CODUFMUN_DESC', 'TPGESTAO_DESC', 'PF_PJ_DESC', 'NIV_DEP_DESC', 'ATIVIDAD_DESC', 'RETENCAO_DESC', 'NATUREZA_DESC', 'TP_UNID_DESC', 'NIV_HIER_DESC', 'TIPEQUIP_DESC', 'IND_SUS_DESC', 'COMPETEN_DESC', 'NAT_JUR_DESC']



,CODUFMUN_DESC,TPGESTAO_DESC,PF_PJ_DESC,NIV_DEP_DESC,ATIVIDAD_DESC,RETENCAO_DESC,NATUREZA_DESC,TP_UNID_DESC,NIV_HIER_DESC,TIPEQUIP_DESC,IND_SUS_DESC,COMPETEN_DESC,NAT_JUR_DESC
0,NaN,NaN,Não informado,Individual,Unidade SEM atividade de Ensino,NaN,NaN,HOSPITAL/DIA - ISOLADO,NaN,.. RAIO X MAIS DE 500MA,NaN,NaN,NaN
1,NaN,NaN,Não informado,Individual,Unidade SEM atividade de Ensino,NaN,NaN,HOSPITAL/DIA - ISOLADO,NaN,.. RAIO X MAIS DE 500MA,NaN,NaN,NaN
2,NaN,NaN,Não informado,Individual,Unidade SEM atividade de Ensino,NaN,NaN,HOSPITAL/DIA - ISOLADO,NaN,.. RAIO X MAIS DE 500MA,NaN,NaN,NaN


## Preview dos Dados Coletados

In [24]:
# Preview do primeiro arquivo SIH gerado
if csvs_sih:
    df_preview_sih = pd.read_csv(csvs_sih[0], nrows=5)
    print(f"SIH — {csvs_sih[0].name}: {df_preview_sih.shape[0]} linhas (amostra) × {df_preview_sih.shape[1]} colunas")
    display(df_preview_sih.head())
else:
    print("Nenhum arquivo SIH encontrado em", PASTA_INPUT_SIH)

SIH — rdsp2501.csv: 5 linhas (amostra) × 113 colunas


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,350000,2025,1,1,46374500028366,3525100117847,1,11704840,354100,19840716,...,NaN,1,0,0,0,0,0,0,0,0
1,350000,2025,1,2,46374500028366,3524130275908,1,11741802,352210,20070606,...,NaN,1,1,1,1,1,0,0,0,0
2,350000,2025,1,2,46374500028366,3524130278427,1,11730000,353110,20030120,...,NaN,1,1,1,0,0,0,0,0,0
3,350000,2025,1,2,46374500028366,3524130278449,1,11743250,352210,20030405,...,NaN,1,1,1,1,0,0,0,0,0
4,350000,2025,1,2,46374500028366,3524130278526,1,11730000,353110,20050224,...,NaN,1,1,1,1,0,0,0,0,0


In [22]:
# Preview do primeiro arquivo CNES (ST) gerado
csvs_cnes_st = sorted(PASTA_INPUT_CNES.glob("st*.csv"))
if csvs_cnes_st:
    df_preview_cnes = pd.read_csv(csvs_cnes_st[0], nrows=5)
    print(f"CNES/ST — {csvs_cnes_st[0].name}: {df_preview_cnes.shape[0]} linhas (amostra) × {df_preview_cnes.shape[1]} colunas")
    display(df_preview_cnes.head())
else:
    print("Nenhum arquivo CNES/ST encontrado em", PASTA_INPUT_CNES)

CNES/ST — stsp2512.csv: 5 linhas (amostra) × 208 colunas


,CNES,CODUFMUN,COD_CEP,CPF_CNPJ,PF_PJ,NIV_DEP,CNPJ_MAN,COD_IR,REGSAUDE,MICR_REG,...,AP07CV02,AP07CV03,AP07CV04,AP07CV05,AP07CV06,AP07CV07,ATEND_PR,DT_ATUAL,COMPETEN,NAT_JUR
0,47406,350010,17800037,35723744000119,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202409,202512,2062
1,81655,350010,17800057,381929000108,3,1,0,NaN,R209,NaN,...,0,0,0,0,0,0,1,202508,202512,2062
2,109789,350010,17803116,36060657000191,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202409,202512,2135
3,109827,350010,17800005,48346595000168,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202409,202512,2135
4,183555,350010,17800043,36363624000110,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202410,202512,2135
